# Final System Audit & Performance Review (FD004)
This evaluation serves as the official integration milestone for the presentation suite. We will isolate the **FD004** dataset (which contains the most aggressively volatile engine life-cycle array in CMAPSS), push it natively through the standalone PINN-LSTM models, evaluate explicit unit-level flight metrics, and deliver a formal RUL degradation scatter map.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf

sys.path.append(os.path.abspath('..'))
from src.data_processor import VectorSequenceProcessor

In [ ]:
# 1. Initialize Processor and Load Volatile Engine Test Sequence
processor = VectorSequenceProcessor()
test_data_path = '../data/test_FD004.txt'

if os.path.exists(test_data_path):
    print("Ingesting FD004 Telemetry Array...")
    df_test = processor.load_data(test_data_path)
    df_test = processor.calculate_rul(df_test)
    df_test = processor.normalize_sensors(df_test, fit=True)
    
    # Generate mapped sequences that carefully track root Engine Unit numbers
    print("Compiling sliding matrices...")
    window_size = 50
    seq_list, truth_list, unit_ids = [], [], []
    
    for unit, group in df_test.groupby('Engine_ID'):
        sens_array = group[processor.sensor_cols].values
        rul_array = group['RUL'].values
        
        for i in range(len(group) - window_size + 1):
            seq_list.append(sens_array[i : i + window_size])
            truth_list.append(rul_array[i + window_size - 1])
            unit_ids.append(unit)
            
    X_test = np.array(seq_list)
    y_test = np.array(truth_list)
    print(f"✅ Successfully staged {X_test.shape[0]} full tracking windows.")
else:
    print(f"⚠️ Error: Could not locate baseline dataset {test_data_path}.")

In [ ]:
# 2. Boot Vector LSTM Neural Core and run mass calculations
model_path = '../models/vxp2_lstm_v1.h5'

if os.path.exists(model_path):
    model = tf.keras.models.load_model(model_path)
    print(f"Executing inference payload against {X_test.shape[0]} targets...")
    
    # Note: Using standard model.predict(), bypassing MC-Dropout strictly for pure accuracy metrics 
    raw_preds = model.predict(X_test, verbose=0).flatten()
    
    overall_rmse = np.sqrt(mean_squared_error(y_test, raw_preds))
    overall_mae = mean_absolute_error(y_test, raw_preds)
    
    print("\n--- GLOBAL PERFORMANCE ---")
    print(f"RMSE (Root Mean Squared Error): {overall_rmse:.2f} Engine Cycles")
    print(f"MAE  (Mean Absolute Error):     {overall_mae:.2f} Engine Cycles")
else:
    print("⚠️ Missing Core Architecture Weights.")

In [ ]:
# 3. Scatter Plot: True RUL degradation mapped against Neural Predictions
if 'raw_preds' in locals():
    plt.figure(figsize=(10, 8))
    sns.set_style("darkgrid")
    
    plt.scatter(y_test, raw_preds, alpha=0.35, color='#2ea043', edgecolors='none', label='Vector Prediction')
    
    # Ideal 1:1 Reference Line
    max_val = max(np.max(y_test), np.max(raw_preds))
    plt.plot([0, max_val], [0, max_val], color='#ff7b72', linestyle='--', linewidth=2, label='Perfect Accuracy (1:1)')
    
    plt.title('Vector VXP2: Explicit Ground Truth vs. Predicted Trajectory', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('True Remaining Useful Life (Cycles)', fontsize=14, labelpad=10)
    plt.ylabel('Vector Predicted Life (Cycles)', fontsize=14, labelpad=10)
    plt.xlim(0, max_val)
    plt.ylim(0, max_val)
    plt.legend(fontsize=12)
    plt.show()

In [ ]:
# 4 & 5. Generate Accuracy Threshold Tables and Qualification Check
if 'raw_preds' in locals():
    results_df = pd.DataFrame({
        'Unit': unit_ids,
        'True_RUL': y_test,
        'Predicted_RUL': raw_preds
    })
    
    # Summarize precision scaling explicitly evaluating mean baseline lifetimes 
    # (~130 cycle bounds average across CMAPSS datasets representations)
    summary_data = []
    max_life_baseline = 130.0 
    
    for u_id, group in results_df.groupby('Unit'):
        mae = mean_absolute_error(group['True_RUL'], group['Predicted_RUL'])
        acc_pct = max(0.0, (1.0 - (mae / max_life_baseline))) * 100.0
        
        summary_data.append({
            'Engine_Unit': u_id,
            'MAE_Cycles': round(mae, 2),
            'Accuracy_%': round(acc_pct, 1)
        })
        
    summary_df = pd.DataFrame(summary_data)
    
    print("--- ENGINE UNIT SUMMARY METRICS (TOP 15 DISPLAYED) ---")
    print(summary_df.head(15).to_string(index=False))
    
    overall_accuracy_rating = summary_df['Accuracy_%'].mean()
    print(f"\n► Overall Simulation Accuracy Baseline: {overall_accuracy_rating:.2f}%")
    
    # Standard Mission Requirement Check (Target: 80%)
    if overall_accuracy_rating > 80.0:
        print("\n🎯 TARGET ACHIEVED: PROTOTYPE 1.0-B QUALIFIED FOR FLIGHT SIMULATION")
    else:
        print("\n⚠️ FLIGHT SIMULATION DELAYED: Architecture requires iterative refinements to breach the 80% ceiling.")